# Lab 2 — Sqoop: importação de clientes e transações

## Objetivo

Este laboratório demonstra a transferência de dados de um banco relacional para a camada Raw de um Data Lake.

Foi utilizada a rota sem privilégios administrativos. O SQLite representa o banco transacional de origem, enquanto a extração por Python representa conceitualmente a importação que seria realizada pelo Sqoop.

A divisão da base de transações em quatro arquivos simula o trabalho de quatro mappers. Entretanto, não existe processamento paralelo real nesta implementação local.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "raw").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "raw").exists():
            pasta_projeto = pasta_pai
            break

pasta_raw = pasta_projeto / "dados" / "raw"
pasta_lab02 = pasta_projeto / "dia1_fundamentos" / "lab02_sqoop"

arquivo_clientes = (
    pasta_raw / "customers" / "customers_synthetic.csv"
)

arquivo_transacoes = (
    pasta_raw / "transactions" / "transactions_synthetic.csv"
)

arquivo_banco = pasta_lab02 / "bigdata_course.db"

print("Projeto:", pasta_projeto)
print("Clientes:", arquivo_clientes)
print("Transações:", arquivo_transacoes)
print("Banco SQLite:", arquivo_banco)

assert arquivo_clientes.exists(), "Arquivo de clientes não encontrado."
assert arquivo_transacoes.exists(), "Arquivo de transações não encontrado."

Projeto: C:\BigData\bigdata-curso-gabriel
Clientes: C:\BigData\bigdata-curso-gabriel\dados\raw\customers\customers_synthetic.csv
Transações: C:\BigData\bigdata-curso-gabriel\dados\raw\transactions\transactions_synthetic.csv
Banco SQLite: C:\BigData\bigdata-curso-gabriel\dia1_fundamentos\lab02_sqoop\bigdata_course.db


In [2]:
clientes_origem = pd.read_csv(arquivo_clientes)
transacoes_origem = pd.read_csv(arquivo_transacoes)

print("Clientes:", len(clientes_origem))
print("Transações:", len(transacoes_origem))

display(clientes_origem.head())
display(transacoes_origem.head())

Clientes: 9993
Transações: 100000


,customer_id,name,cpf,email,segment,credit_score,created_at
0,1,Ana Laura Campos,943.065.218-42,igor46@example.com,Premium,426,2026-07-03
1,2,Mariah Caldeira,586.237.094-38,jose48@example.com,High-Risk,481,2026-06-17
2,3,Kevin Cavalcante,530.629.814-15,theoda-costa@example.org,Standard,708,2025-11-06
3,4,Maria Laura Freitas,725.130.864-90,castrolucas@example.org,Premium,473,2026-02-07
4,5,Srta. Mirella Moura,570.814.239-14,emilly20@example.com,Premium,816,2025-10-04


,transaction_id,customer_id,amount,transaction_type,timestamp,status,risk_score,is_fraud
0,9157,4627,156.683351,pagamento,2023-01-01,approved,89.859250,False
1,46882,9378,61.694980,compra,2023-01-01,approved,64.229229,False
2,91656,1385,41.844959,transferencia,2023-01-01,approved,0.183873,False
3,22248,8256,46.843159,pagamento,2023-01-01,declined,87.757724,False
4,15064,229,48.793758,compra,2023-01-01,approved,19.456056,False


In [3]:
with sqlite3.connect(arquivo_banco) as conexao:
    clientes_origem.to_sql(
        "customers",
        conexao,
        if_exists="replace",
        index=False
    )

    transacoes_origem.to_sql(
        "transactions",
        conexao,
        if_exists="replace",
        index=False
    )

print("Banco SQLite criado com sucesso.")
print("Tabelas criadas: customers e transactions.")

Banco SQLite criado com sucesso.
Tabelas criadas: customers e transactions.


In [4]:
with sqlite3.connect(arquivo_banco) as conexao:
    tabelas = pd.read_sql_query(
        """
        SELECT name AS tabela
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        conexao
    )

    quantidade_clientes = conexao.execute(
        "SELECT COUNT(*) FROM customers"
    ).fetchone()[0]

    quantidade_transacoes = conexao.execute(
        "SELECT COUNT(*) FROM transactions"
    ).fetchone()[0]

display(tabelas)

print("Clientes no SQLite:", quantidade_clientes)
print("Transações no SQLite:", quantidade_transacoes)

,tabela
0,customers
1,transactions


Clientes no SQLite: 9993
Transações no SQLite: 100000


In [5]:
with sqlite3.connect(arquivo_banco) as conexao:
    schema_clientes = pd.read_sql_query(
        "PRAGMA table_info(customers)",
        conexao
    )

    schema_transacoes = pd.read_sql_query(
        "PRAGMA table_info(transactions)",
        conexao
    )

print("Schema da tabela customers:")
display(schema_clientes[["name", "type"]])

print("Schema da tabela transactions:")
display(schema_transacoes[["name", "type"]])

Schema da tabela customers:


,name,type
0,customer_id,INTEGER
1,name,TEXT
2,cpf,TEXT
3,email,TEXT
4,segment,TEXT
5,credit_score,INTEGER
6,created_at,TEXT


Schema da tabela transactions:


,name,type
0,transaction_id,INTEGER
1,customer_id,INTEGER
2,amount,REAL
3,transaction_type,TEXT
4,timestamp,TEXT
5,status,TEXT
6,risk_score,REAL
7,is_fraud,INTEGER


In [6]:
pasta_importacao = pasta_raw / "sqoop_import"
pasta_importacao_clientes = pasta_importacao / "customers"
pasta_importacao_transacoes = pasta_importacao / "transactions"

pasta_importacao_clientes.mkdir(parents=True, exist_ok=True)
pasta_importacao_transacoes.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(arquivo_banco) as conexao:
    clientes_importados = pd.read_sql_query(
        "SELECT * FROM customers",
        conexao
    )

    transacoes_importadas = pd.read_sql_query(
        "SELECT * FROM transactions",
        conexao
    )

clientes_importados.to_csv(
    pasta_importacao_clientes / "customers_from_db.csv",
    index=False
)

transacoes_importadas.to_csv(
    pasta_importacao_transacoes / "transactions_from_db.csv",
    index=False
)

print("Importação concluída.")
print("Clientes importados:", len(clientes_importados))
print("Transações importadas:", len(transacoes_importadas))

Importação concluída.
Clientes importados: 9993
Transações importadas: 100000


In [7]:
validacao_importacao = pd.DataFrame({
    "Base": ["Clientes", "Transações"],
    "Linhas na origem": [
        len(clientes_origem),
        len(transacoes_origem)
    ],
    "Linhas no SQLite": [
        quantidade_clientes,
        quantidade_transacoes
    ],
    "Linhas importadas": [
        len(clientes_importados),
        len(transacoes_importadas)
    ]
})

validacao_importacao["Validação"] = (
    (validacao_importacao["Linhas na origem"]
     == validacao_importacao["Linhas no SQLite"])
    &
    (validacao_importacao["Linhas no SQLite"]
     == validacao_importacao["Linhas importadas"])
).map({
    True: "OK",
    False: "DIVERGENTE"
})

validacao_importacao

,Base,Linhas na origem,Linhas no SQLite,Linhas importadas,Validação
0,Clientes,9993,9993,9993,OK
1,Transações,100000,100000,100000,OK


In [8]:
pasta_mappers = pasta_raw / "_mapper_demo"
pasta_mappers.mkdir(parents=True, exist_ok=True)

resultado_mappers = []

with sqlite3.connect(arquivo_banco) as conexao:
    for numero_mapper in range(4):
        consulta = f"""
            SELECT *
            FROM transactions
            WHERE transaction_id % 4 = {numero_mapper}
            ORDER BY transaction_id
        """

        parte = pd.read_sql_query(consulta, conexao)

        arquivo_parte = (
            pasta_mappers / f"part-{numero_mapper:05d}.csv"
        )

        parte.to_csv(arquivo_parte, index=False)

        resultado_mappers.append({
            "Mapper": numero_mapper,
            "Arquivo": arquivo_parte.name,
            "Linhas": len(parte)
        })

resultado_mappers_df = pd.DataFrame(resultado_mappers)
resultado_mappers_df

,Mapper,Arquivo,Linhas
0,0,part-00000.csv,25000
1,1,part-00001.csv,25000
2,2,part-00002.csv,25000
3,3,part-00003.csv,25000


In [9]:
ids_mappers = []

for arquivo in sorted(pasta_mappers.glob("part-*.csv")):
    parte = pd.read_csv(arquivo)
    ids_mappers.extend(parte["transaction_id"].tolist())

validacao_mappers = pd.DataFrame({
    "Indicador": [
        "Total de linhas nas quatro partes",
        "Identificadores únicos",
        "Identificadores duplicados",
        "Total esperado"
    ],
    "Resultado": [
        len(ids_mappers),
        len(set(ids_mappers)),
        len(ids_mappers) - len(set(ids_mappers)),
        len(transacoes_origem)
    ]
})

validacao_mappers

,Indicador,Resultado
0,Total de linhas nas quatro partes,100000
1,Identificadores únicos,100000
2,Identificadores duplicados,0
3,Total esperado,100000


## Conclusão

As bases de clientes e transações foram carregadas em um banco relacional SQLite, representando o sistema de origem. As tabelas criadas continham, respectivamente, 9.993 clientes e 100.000 transações.

Em seguida, os dados foram extraídos do SQLite e armazenados na camada Raw. As contagens foram comparadas entre os arquivos de origem, o banco relacional e os arquivos importados, sem perda ou duplicação de registros.

A base de transações também foi dividida em quatro arquivos com 25.000 registros cada. Essa divisão representa conceitualmente a atuação de quatro mappers em uma importação realizada pelo Sqoop. Na solução local, contudo, os arquivos foram gerados sequencialmente e não por processamento paralelo distribuído.